# Práctica 1 - Sistema híbrido de emparejamiento interpersonal
Introducción a la Inteligencia Artificial (3010476)

El sistema recibe una ontología (`.ttl`) con personas y su información, y para cada par de personas decide una acción:
`proponer_cita` (con lugar sugerido), `sugerir_conversar`, `pedir_verificacion`, `descartar` o `revisar_manual`.

**Flujo general**
1. **Ontología**: se carga `emparejamiento.ttl` y se razona con `DeductiveClosure(RDFS_Semantics)` de OWL-RL.
2. **Traductor**: todo el grafo inferido se convierte en hechos genéricos de Experta (`Individuo`, `Relacion`, `Propiedad`, `Esquema`).
3. **Lógica difusa**: por cada par se calcula diferencia de edad, afinidad de intereses (Jaccard) y distancia (Haversine) y se obtiene la compatibilidad.
4. **Sistema experto**: recibe los hechos de la ontología y los `AfinidadDifusa`, y con sus reglas toma la decisión.

Cada módulo se escribe como archivo `.py` con `%%writefile` y al final se ejecuta `main.py`.

## Instalación de librerías

In [ ]:
!pip install -q rdflib==7.6.0 owlrl==7.6.2 scikit-fuzzy==0.5.0 experta==1.9.4
!mkdir -p ontologia difuso experto integracion docs
!touch ontologia/__init__.py difuso/__init__.py experto/__init__.py integracion/__init__.py

## Vocabulario del dominio
Constantes con los URIs que usan las reglas y la preparación del difuso. El traductor no lo usa.

In [ ]:
%%writefile vocab.py
"""
Constantes con los nombres del dominio de emparejamiento.

Las usan el sistema experto, la preparación del difuso y el main.
El traductor NO importa este archivo, para que no dependa del dominio.
"""

MT = "http://ejemplo.org/emparejamiento#"
FOAF = "http://xmlns.com/foaf/0.1/"
RDFS = "http://www.w3.org/2000/01/rdf-schema#"

# Clases
MT_PERSONA = MT + "Persona"
MT_ESTUDIANTE = MT + "Estudiante"
MT_PROFESIONAL = MT + "Profesional"
MT_INTERES = MT + "Interes"
MT_DEPORTE = MT + "Deporte"
MT_VALOR = MT + "Valor"
MT_OBJETIVO = MT + "ObjetivoRelacion"
MT_LUGAR = MT + "Lugar"
MT_CIUDAD = MT + "Ciudad"
MT_LUGAR_CITA = MT + "LugarDeCita"
MT_CITA = MT + "Cita"

# Propiedades
MT_TIENE_INTERES = MT + "tieneInteres"
MT_PRACTICA_DEPORTE = MT + "practicaDeporte"
MT_TIENE_VALOR = MT + "tieneValor"
MT_BUSCA_OBJETIVO = MT + "buscaObjetivo"
MT_INCOMPATIBLE = MT + "esIncompatibleCon"
MT_VIVE_EN = MT + "viveEn"
MT_AMIGO_DE = MT + "esAmigoDe"
MT_PARTICIPA_EN = MT + "participaEn"
MT_OCURRIO_EN = MT + "ocurrioEn"
MT_UBICADO_EN = MT + "ubicadoEn"
MT_EDAD = MT + "edad"
MT_CALIFICACION = MT + "calificacion"
MT_LATITUD = MT + "latitud"
MT_LONGITUD = MT + "longitud"
MT_VERIFICADO = MT + "perfilVerificado"

FOAF_PERSON = FOAF + "Person"
FOAF_NAME = FOAF + "name"
FOAF_KNOWS = FOAF + "knows"
RDFS_LABEL = RDFS + "label"
RDFS_CLASS = RDFS + "Class"

# Relaciones de esquema (campo 'relacion' del Fact Esquema)
SUBCLASE = "subClassOf"
SUBPROPIEDAD = "subPropertyOf"
DOMINIO = "domain"
RANGO = "range"
TIPO = "tipo"


def nombre_corto(uri):
    """Devuelve la parte final de un URI (lo que va despues de # o /)."""
    return uri.split("#")[-1].split("/")[-1]

# Módulo 1: Ontología
## Ontología en Turtle
11 clases, 6 `subClassOf`, 15 propiedades propias con dominio y rango, `subPropertyOf`, vocabulario FOAF y 4 individuos por clase. `Camila` no tiene `rdf:type` a propósito.

In [ ]:
%%writefile ontologia/emparejamiento.ttl
@prefix mt:   <http://ejemplo.org/emparejamiento#> .
@prefix foaf: <http://xmlns.com/foaf/0.1/> .
@prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .

#################################################################
# Clases
#################################################################

mt:Persona a rdfs:Class ;
    rdfs:subClassOf foaf:Person ;
    rdfs:label "Persona"@es .

mt:Estudiante a rdfs:Class ;
    rdfs:subClassOf mt:Persona ;
    rdfs:label "Estudiante"@es .

mt:Profesional a rdfs:Class ;
    rdfs:subClassOf mt:Persona ;
    rdfs:label "Profesional"@es .

mt:Interes a rdfs:Class ;
    rdfs:label "Interés o afición"@es .

mt:Deporte a rdfs:Class ;
    rdfs:subClassOf mt:Interes ;
    rdfs:label "Deporte"@es .

mt:Valor a rdfs:Class ;
    rdfs:label "Valor personal"@es .

mt:ObjetivoRelacion a rdfs:Class ;
    rdfs:label "Objetivo de relación"@es .

mt:Lugar a rdfs:Class ;
    rdfs:label "Lugar"@es .

mt:Ciudad a rdfs:Class ;
    rdfs:subClassOf mt:Lugar ;
    rdfs:label "Ciudad"@es .

mt:LugarDeCita a rdfs:Class ;
    rdfs:subClassOf mt:Lugar ;
    rdfs:label "Lugar para una cita"@es .

mt:Cita a rdfs:Class ;
    rdfs:label "Cita realizada"@es .

#################################################################
# Propiedades
#################################################################

mt:tieneInteres a rdf:Property ;
    rdfs:domain mt:Persona ; rdfs:range mt:Interes ;
    rdfs:label "tiene interés"@es .

mt:practicaDeporte a rdf:Property ;
    rdfs:subPropertyOf mt:tieneInteres ;
    rdfs:domain mt:Persona ; rdfs:range mt:Deporte ;
    rdfs:label "practica deporte"@es .

mt:tieneValor a rdf:Property ;
    rdfs:domain mt:Persona ; rdfs:range mt:Valor ;
    rdfs:label "tiene valor"@es .

mt:buscaObjetivo a rdf:Property ;
    rdfs:domain mt:Persona ; rdfs:range mt:ObjetivoRelacion ;
    rdfs:label "busca objetivo"@es .

mt:esIncompatibleCon a rdf:Property ;
    rdfs:domain mt:ObjetivoRelacion ; rdfs:range mt:ObjetivoRelacion ;
    rdfs:label "es incompatible con"@es .

mt:viveEn a rdf:Property ;
    rdfs:domain mt:Persona ; rdfs:range mt:Ciudad ;
    rdfs:label "vive en"@es .

mt:esAmigoDe a rdf:Property ;
    rdfs:subPropertyOf foaf:knows ;
    rdfs:domain mt:Persona ; rdfs:range mt:Persona ;
    rdfs:label "es amigo de"@es .

mt:participaEn a rdf:Property ;
    rdfs:domain mt:Persona ; rdfs:range mt:Cita ;
    rdfs:label "participa en"@es .

mt:ocurrioEn a rdf:Property ;
    rdfs:domain mt:Cita ; rdfs:range mt:LugarDeCita ;
    rdfs:label "ocurrió en"@es .

mt:ubicadoEn a rdf:Property ;
    rdfs:domain mt:Lugar ; rdfs:range mt:Lugar ;
    rdfs:label "ubicado en"@es .

mt:edad a rdf:Property ;
    rdfs:domain mt:Persona ; rdfs:range xsd:integer ;
    rdfs:label "edad"@es .

mt:calificacion a rdf:Property ;
    rdfs:domain mt:Cita ; rdfs:range xsd:integer ;
    rdfs:label "calificación de la cita (1 a 5)"@es .

mt:latitud a rdf:Property ;
    rdfs:domain mt:Ciudad ; rdfs:range xsd:decimal ;
    rdfs:label "latitud"@es .

mt:longitud a rdf:Property ;
    rdfs:domain mt:Ciudad ; rdfs:range xsd:decimal ;
    rdfs:label "longitud"@es .

mt:perfilVerificado a rdf:Property ;
    rdfs:domain mt:Persona ; rdfs:range xsd:boolean ;
    rdfs:label "perfil verificado"@es .

#################################################################
# Intereses y deportes
#################################################################

mt:Cine a mt:Interes ; rdfs:label "Cine"@es .
mt:Lectura a mt:Interes ; rdfs:label "Lectura"@es .
mt:Musica a mt:Interes ; rdfs:label "Música"@es .
mt:Viajar a mt:Interes ; rdfs:label "Viajar"@es .

mt:Futbol a mt:Deporte ; rdfs:label "Fútbol"@es .
mt:Running a mt:Deporte ; rdfs:label "Running"@es .
mt:Ciclismo a mt:Deporte ; rdfs:label "Ciclismo"@es .
mt:Natacion a mt:Deporte ; rdfs:label "Natación"@es .

#################################################################
# Valores y objetivos
#################################################################

mt:Honestidad a mt:Valor ; rdfs:label "Honestidad"@es .
mt:Familia a mt:Valor ; rdfs:label "Familia"@es .
mt:Aventura a mt:Valor ; rdfs:label "Aventura"@es .
mt:Ambicion a mt:Valor ; rdfs:label "Ambición"@es .

mt:Amistad a mt:ObjetivoRelacion ; rdfs:label "Amistad"@es .
mt:RelacionCasual a mt:ObjetivoRelacion ; rdfs:label "Relación casual"@es .
mt:RelacionSeria a mt:ObjetivoRelacion ; rdfs:label "Relación seria"@es .
mt:Matrimonio a mt:ObjetivoRelacion ; rdfs:label "Matrimonio"@es .

# RDFS no maneja propiedades simétricas, por eso van en los dos sentidos
mt:RelacionSeria mt:esIncompatibleCon mt:RelacionCasual .
mt:RelacionCasual mt:esIncompatibleCon mt:RelacionSeria .
mt:Matrimonio mt:esIncompatibleCon mt:RelacionCasual .
mt:RelacionCasual mt:esIncompatibleCon mt:Matrimonio .

#################################################################
# Lugares
#################################################################

mt:Antioquia a mt:Lugar ; rdfs:label "Antioquia"@es .
mt:AreaMetropolitana a mt:Lugar ; rdfs:label "Área Metropolitana"@es ;
    mt:ubicadoEn mt:Antioquia .
mt:OrienteAntioqueno a mt:Lugar ; rdfs:label "Oriente Antioqueño"@es ;
    mt:ubicadoEn mt:Antioquia .
mt:EjeCafetero a mt:Lugar ; rdfs:label "Eje Cafetero"@es .

mt:Medellin a mt:Ciudad ; rdfs:label "Medellín"@es ;
    mt:latitud 6.2442 ; mt:longitud -75.5812 ;
    mt:ubicadoEn mt:AreaMetropolitana .
mt:Envigado a mt:Ciudad ; rdfs:label "Envigado"@es ;
    mt:latitud 6.1759 ; mt:longitud -75.5917 ;
    mt:ubicadoEn mt:AreaMetropolitana .
mt:Rionegro a mt:Ciudad ; rdfs:label "Rionegro"@es ;
    mt:latitud 6.1550 ; mt:longitud -75.3737 ;
    mt:ubicadoEn mt:OrienteAntioqueno .
mt:Manizales a mt:Ciudad ; rdfs:label "Manizales"@es ;
    mt:latitud 5.0703 ; mt:longitud -75.5138 ;
    mt:ubicadoEn mt:EjeCafetero .

mt:JardinBotanico a mt:LugarDeCita ; rdfs:label "Jardín Botánico"@es ;
    mt:ubicadoEn mt:Medellin .
mt:ParqueExplora a mt:LugarDeCita ; rdfs:label "Parque Explora"@es ;
    mt:ubicadoEn mt:Medellin .
mt:ParqueLleras a mt:LugarDeCita ; rdfs:label "Parque Lleras"@es ;
    mt:ubicadoEn mt:Medellin .
mt:CafeCentral a mt:LugarDeCita ; rdfs:label "Café Central"@es ;
    mt:ubicadoEn mt:Envigado .

#################################################################
# Citas pasadas
#################################################################

mt:cita1 a mt:Cita ; rdfs:label "Cita 1"@es ;
    mt:ocurrioEn mt:ParqueLleras ; mt:calificacion 2 .
mt:cita2 a mt:Cita ; rdfs:label "Cita 2"@es ;
    mt:ocurrioEn mt:CafeCentral ; mt:calificacion 4 .
mt:cita3 a mt:Cita ; rdfs:label "Cita 3"@es ;
    mt:ocurrioEn mt:JardinBotanico ; mt:calificacion 5 .
mt:cita4 a mt:Cita ; rdfs:label "Cita 4"@es ;
    mt:ocurrioEn mt:ParqueExplora ; mt:calificacion 3 .

#################################################################
# Personas
#################################################################

# --- Personas sin subtipo ---
mt:Laura a mt:Persona ;
    foaf:name "Laura" ; mt:edad 29 ; mt:viveEn mt:Envigado ;
    mt:tieneInteres mt:Viajar, mt:Musica ; mt:practicaDeporte mt:Natacion ;
    mt:tieneValor mt:Aventura ; mt:buscaObjetivo mt:Amistad ;
    mt:esAmigoDe mt:Sofia, mt:Andres ;
    mt:participaEn mt:cita2 ; mt:perfilVerificado true .

mt:Carlos a mt:Persona ;
    foaf:name "Carlos" ; mt:edad 31 ; mt:viveEn mt:Envigado ;
    mt:tieneInteres mt:Viajar ; mt:practicaDeporte mt:Ciclismo ;
    mt:tieneValor mt:Aventura, mt:Ambicion ; mt:buscaObjetivo mt:Amistad ;
    mt:esAmigoDe mt:Julian ;
    mt:participaEn mt:cita2 ; mt:perfilVerificado true .

mt:Diana a mt:Persona ;
    foaf:name "Diana" ; mt:edad 35 ; mt:viveEn mt:Rionegro ;
    mt:tieneInteres mt:Lectura ; mt:practicaDeporte mt:Ciclismo ;
    mt:tieneValor mt:Familia ; mt:buscaObjetivo mt:Matrimonio ;
    mt:esAmigoDe mt:Julian ;
    mt:participaEn mt:cita3 ; mt:perfilVerificado true .

mt:Julian a mt:Persona ;
    foaf:name "Julián" ; mt:edad 38 ; mt:viveEn mt:Rionegro ;
    mt:tieneInteres mt:Lectura, mt:Cine ; mt:practicaDeporte mt:Natacion ;
    mt:tieneValor mt:Familia ; mt:buscaObjetivo mt:Matrimonio ;
    mt:esAmigoDe mt:Carlos, mt:Diana ;
    mt:participaEn mt:cita3 ; mt:perfilVerificado false .

# --- Estudiantes ---
mt:Sofia a mt:Estudiante ;
    foaf:name "Sofía" ; mt:edad 23 ; mt:viveEn mt:Medellin ;
    mt:tieneInteres mt:Cine, mt:Lectura ; mt:practicaDeporte mt:Running ;
    mt:tieneValor mt:Honestidad ; mt:buscaObjetivo mt:RelacionSeria ;
    mt:esAmigoDe mt:Laura ;
    mt:perfilVerificado true .

mt:Tomas a mt:Estudiante ;
    foaf:name "Tomás" ; mt:edad 21 ; mt:viveEn mt:Medellin ;
    mt:tieneInteres mt:Musica, mt:Viajar ; mt:practicaDeporte mt:Futbol ;
    mt:tieneValor mt:Aventura ; mt:buscaObjetivo mt:RelacionCasual ;
    mt:participaEn mt:cita1 ; mt:perfilVerificado true .

mt:Isabela a mt:Estudiante ;
    foaf:name "Isabela" ; mt:edad 22 ; mt:viveEn mt:Medellin ;
    mt:tieneInteres mt:Cine, mt:Viajar ; mt:practicaDeporte mt:Natacion ;
    mt:tieneValor mt:Honestidad ; mt:buscaObjetivo mt:Amistad ;
    mt:participaEn mt:cita1 ; mt:perfilVerificado true .

mt:Samuel a mt:Estudiante ;
    foaf:name "Samuel" ; mt:edad 24 ; mt:viveEn mt:Envigado ;
    mt:tieneInteres mt:Musica ; mt:practicaDeporte mt:Futbol, mt:Running ;
    mt:tieneValor mt:Ambicion ; mt:buscaObjetivo mt:RelacionCasual ;
    mt:participaEn mt:cita4 ; mt:perfilVerificado true .

# --- Profesionales ---
mt:Andres a mt:Profesional ;
    foaf:name "Andrés" ; mt:edad 26 ; mt:viveEn mt:Medellin ;
    mt:tieneInteres mt:Cine, mt:Lectura, mt:Musica ; mt:practicaDeporte mt:Running ;
    mt:tieneValor mt:Honestidad ; mt:buscaObjetivo mt:RelacionSeria ;
    mt:esAmigoDe mt:Laura ;
    mt:perfilVerificado true .

mt:Mateo a mt:Profesional ;
    foaf:name "Mateo" ; mt:edad 41 ; mt:viveEn mt:Manizales ;
    mt:tieneInteres mt:Musica ; mt:practicaDeporte mt:Futbol ;
    mt:tieneValor mt:Ambicion ; mt:buscaObjetivo mt:RelacionCasual ;
    mt:perfilVerificado true .

mt:Paula a mt:Profesional ;
    foaf:name "Paula" ; mt:edad 28 ; mt:viveEn mt:Medellin ;
    mt:tieneInteres mt:Lectura, mt:Musica ; mt:practicaDeporte mt:Ciclismo ;
    mt:tieneValor mt:Familia ; mt:buscaObjetivo mt:RelacionSeria ;
    mt:perfilVerificado true .

mt:Daniel a mt:Profesional ;
    foaf:name "Daniel" ; mt:edad 33 ; mt:viveEn mt:Manizales ;
    mt:tieneInteres mt:Viajar ; mt:practicaDeporte mt:Futbol ;
    mt:tieneValor mt:Honestidad ; mt:buscaObjetivo mt:Matrimonio ;
    mt:participaEn mt:cita4 ; mt:perfilVerificado true .

# --- Sin rdf:type a propósito: el razonador debe deducir que es Persona ---
mt:Camila foaf:name "Camila" ; mt:edad 27 ; mt:viveEn mt:Envigado ;
    mt:tieneInteres mt:Cine, mt:Musica ; mt:practicaDeporte mt:Running ;
    mt:tieneValor mt:Honestidad ; mt:buscaObjetivo mt:RelacionSeria ;
    mt:perfilVerificado false .

## Razonamiento RDFS
Carga el grafo, aplica el razonador y muestra los tres casos de inferencia y la comparación antes/después.

In [ ]:
%%writefile ontologia/razonamiento.py
"""
Carga de la ontologia y razonamiento RDFS con OWL-RL.
"""
from rdflib import Graph, RDF, RDFS, Namespace
from owlrl import DeductiveClosure, RDFS_Semantics

MT = Namespace("http://ejemplo.org/emparejamiento#")


def cargar_y_razonar(ruta_ttl):
    """Devuelve (grafo_original, grafo_inferido) tras DeductiveClosure(RDFS_Semantics).

    Se parsea el archivo dos veces porque el razonador modifica el grafo
    sobre el que trabaja, y el original se necesita para saber que es inferido.
    """
    g_original = Graph()
    g_original.parse(ruta_ttl, format="turtle")

    g_inferido = Graph()
    g_inferido.parse(ruta_ttl, format="turtle")
    DeductiveClosure(RDFS_Semantics).expand(g_inferido)
    return g_original, g_inferido


def nuevos_hechos(g_original, g_inferido):
    """Triples que estan en el grafo inferido y no en el original,
    dejando solo los que tienen sujeto del namespace mt."""
    return [t for t in g_inferido
            if t not in g_original and str(t[0]).startswith(str(MT))]


def casos_inferencia(g_original, g_inferido):
    """Arma los ejemplos de los tres casos pedidos a partir de lo inferido.

    caso 1: tipo por jerarquia de clases (subClassOf)
    caso 2: tipo por dominio/rango de una propiedad (individuo sin rdf:type)
    caso 3: nuevo hecho por subPropertyOf
    """
    nuevos = set(nuevos_hechos(g_original, g_inferido))

    caso1 = []
    for s, _, c in g_original.triples((None, RDF.type, None)):
        for sup in g_original.objects(c, RDFS.subClassOf):
            if (s, RDF.type, sup) in nuevos:
                caso1.append((s, RDF.type, sup))

    caso2 = []
    for s in set(g_original.subjects()):
        if (s, RDF.type, None) in g_original:
            continue
        for p in set(g_original.predicates(s, None)):
            for c in g_original.objects(p, RDFS.domain):
                if (s, RDF.type, c) in nuevos:
                    caso2.append((s, RDF.type, c, p))

    caso3 = []
    for p, sup in g_original.subject_objects(RDFS.subPropertyOf):
        for s, o in g_original.subject_objects(p):
            if (s, sup, o) in nuevos:
                caso3.append((s, sup, o, p))

    return caso1, caso2, caso3


def mostrar_razonamiento(g_original, g_inferido):
    """Imprime la comparacion antes/despues y los casos de inferencia."""
    g_inferido.bind("mt", MT)
    q = g_inferido.namespace_manager.normalizeUri

    print(f"Triples antes del razonamiento:   {len(g_original)}")
    print(f"Triples despues del razonamiento: {len(g_inferido)}")
    nuevos = nuevos_hechos(g_original, g_inferido)
    print(f"Triples nuevos sobre recursos mt: {len(nuevos)}")

    caso1, caso2, caso3 = casos_inferencia(g_original, g_inferido)

    print("\nCaso 1 - jerarquia de clases (subClassOf):")
    for s, p, o in sorted(caso1)[:8]:
        print(f"   {q(s)} a {q(o)}")
    print(f"   ... total {len(caso1)}")

    print("\nCaso 2 - pertenencia desde el dominio de una propiedad:")
    for s, p, o, prop in caso2:
        print(f"   {q(s)} a {q(o)}   (por el dominio de {q(prop)})")

    print("\nCaso 3 - jerarquia de propiedades (subPropertyOf):")
    for s, p, o, prop in sorted(caso3):
        print(f"   {q(s)} {q(p)} {q(o)}   (desde {q(prop)})")

    print("\nOtros hechos nuevos (muestra):")
    ya_mostrados = set(caso1) | {(s, p, o) for s, p, o, _ in caso2 + caso3}
    otros = [t for t in nuevos if t[1] == RDF.type and t[2] != RDFS.Resource
             and t not in ya_mostrados]
    for s, p, o in sorted(otros)[:8]:
        print(f"   {q(s)} a {q(o)}")

## Diagrama de clases

In [ ]:
%%writefile ontologia/diagrama.py
"""
Dibuja el diagrama de clases de la ontologia leyendo el .ttl.
Flechas solidas: rdfs:subClassOf. Flechas punteadas: propiedades entre clases (dominio -> rango).
"""
import os

import matplotlib
import matplotlib.pyplot as plt
from rdflib import Graph, RDF, RDFS

# posicion de cada clase en el dibujo (solo para que se vea ordenado)
POSICIONES = {
    "Person": (1.0, 9.0), "Persona": (1.0, 7.0),
    "Estudiante": (0.0, 5.0), "Profesional": (2.0, 5.0),
    "Interes": (6.0, 9.0), "Deporte": (6.0, 6.0),
    "Valor": (9.5, 9.0), "ObjetivoRelacion": (9.5, 4.5),
    "Lugar": (6.0, 3.0), "Ciudad": (4.5, 1.0), "LugarDeCita": (7.5, 1.0),
    "Cita": (1.0, 2.5),
}


def corto(uri):
    """Nombre local de un URI."""
    return str(uri).split("#")[-1].split("/")[-1]


def dibujar_diagrama(ruta_ttl, carpeta="docs"):
    """Genera docs/diagrama_clases.png a partir de la ontologia."""
    g = Graph()
    g.parse(ruta_ttl, format="turtle")
    fig, ax = plt.subplots(figsize=(14, 10))
    ax.axis("off")

    for nombre, (x, y) in POSICIONES.items():
        ax.text(x, y, nombre, ha="center", va="center", fontsize=11,
                bbox=dict(boxstyle="round,pad=0.4", fc="#dbe9f6", ec="#2b5d8a"))

    def flecha(o, d, estilo, color, texto=""):
        """Dibuja una flecha de la clase o a la clase d (o un bucle si son iguales)."""
        (x1, y1), (x2, y2) = POSICIONES[o], POSICIONES[d]
        if o == d:
            ax.annotate("", xy=(x1 - 0.2, y1 + 0.2), xytext=(x1 + 0.2, y1 + 0.2),
                        arrowprops=dict(arrowstyle="->", color=color, ls=estilo,
                                        connectionstyle="arc3,rad=1.8"))
            ax.text(x1, y1 + 0.75, texto, fontsize=8, color=color, ha="center")
            return
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                    arrowprops=dict(arrowstyle="->", color=color, ls=estilo,
                                    shrinkA=18, shrinkB=18))
        if texto:
            ax.text((x1 + x2) / 2, (y1 + y2) / 2, texto, fontsize=8, color=color,
                    ha="center", bbox=dict(fc="white", ec="none", pad=0.5))

    for s, o in g.subject_objects(RDFS.subClassOf):
        if corto(s) in POSICIONES and corto(o) in POSICIONES:
            flecha(corto(s), corto(o), "-", "black", "subClassOf")

    for p in g.subjects(RDF.type, RDF.Property):
        dom, ran = g.value(p, RDFS.domain), g.value(p, RDFS.range)
        if dom is None or ran is None:
            continue
        if corto(dom) in POSICIONES and corto(ran) in POSICIONES:
            flecha(corto(dom), corto(ran), "--", "#b0501c", corto(p))

    # propiedades con rango literal, listadas bajo su clase
    literales = {}
    for p in g.subjects(RDF.type, RDF.Property):
        dom, ran = g.value(p, RDFS.domain), g.value(p, RDFS.range)
        if ran is not None and "XMLSchema" in str(ran):
            literales.setdefault(corto(dom), []).append(f"{corto(p)}: xsd:{corto(ran)}")
    for clase, props in literales.items():
        x, y = POSICIONES[clase]
        ax.text(x, y - 0.45, "\n".join(props), ha="center", va="top", fontsize=7.5, color="#444")

    ax.set_xlim(-1, 11)
    ax.set_ylim(0, 10)
    ax.set_title("Ontología de emparejamiento (mt:)")
    os.makedirs(carpeta, exist_ok=True)
    ruta = os.path.join(carpeta, "diagrama_clases.png")
    fig.savefig(ruta, dpi=120, bbox_inches="tight")
    plt.close(fig)
    return ruta


if __name__ == "__main__":
    matplotlib.use("Agg")
    base = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    print(dibujar_diagrama(os.path.join(base, "ontologia", "emparejamiento.ttl"),
                           os.path.join(base, "docs")))

# Módulo 2: Lógica difusa
## Variables y funciones de pertenencia
Incluye funciones triangulares, trapezoidales y gaussianas, y los modificadores *muy* y *ligeramente*.

In [ ]:
%%writefile difuso/variables.py
"""
Variables difusas, universos y funciones de pertenencia.
"""
import numpy as np
import skfuzzy as fuzz
from skfuzzy import control as ctrl


def muy(mf):
    """Modificador 'muy' (concentracion): mu(x)^2."""
    return np.power(mf, 2)


def ligeramente(mf):
    """Modificador 'ligeramente' (dilatacion): mu(x)^0.5."""
    return np.power(mf, 0.5)


def crear_variables():
    """Crea las 3 entradas y la salida con sus valores linguisticos.

    Devuelve (diferencia_edad, afinidad_intereses, distancia_km, compatibilidad).
    """
    # Universos del discurso
    diferencia_edad = ctrl.Antecedent(np.arange(0, 30.1, 0.1), "diferencia_edad")       # años
    afinidad_intereses = ctrl.Antecedent(np.arange(0, 100.1, 0.5), "afinidad_intereses")  # %
    distancia_km = ctrl.Antecedent(np.arange(0, 150.1, 0.5), "distancia_km")              # km
    compatibilidad = ctrl.Consequent(np.arange(0, 100.1, 0.5), "compatibilidad")

    u = diferencia_edad.universe
    diferencia_edad["pequena"] = fuzz.trapmf(u, [0, 0, 3, 7])
    diferencia_edad["moderada"] = fuzz.trimf(u, [4, 9, 14])
    diferencia_edad["grande"] = fuzz.trapmf(u, [11, 18, 30, 30])

    u = afinidad_intereses.universe
    afinidad_intereses["baja"] = fuzz.trapmf(u, [0, 0, 15, 35])
    afinidad_intereses["media"] = fuzz.gaussmf(u, 45, 12)
    afinidad_intereses["alta"] = fuzz.trapmf(u, [50, 70, 100, 100])
    # modificador aplicado como un termino nuevo
    afinidad_intereses["muy_alta"] = muy(afinidad_intereses["alta"].mf)

    u = distancia_km.universe
    distancia_km["cerca"] = fuzz.gaussmf(u, 0, 10)
    distancia_km["intermedia"] = fuzz.trimf(u, [10, 40, 80])
    distancia_km["lejos"] = fuzz.trapmf(u, [60, 100, 150, 150])
    distancia_km["ligeramente_lejos"] = ligeramente(distancia_km["lejos"].mf)

    u = compatibilidad.universe
    compatibilidad["baja"] = fuzz.trimf(u, [0, 0, 45])
    compatibilidad["media"] = fuzz.gaussmf(u, 50, 12)
    compatibilidad["alta"] = fuzz.trimf(u, [55, 100, 100])
    compatibilidad.defuzzify_method = "centroid"

    return diferencia_edad, afinidad_intereses, distancia_km, compatibilidad

## Reglas difusas
11 reglas con AND (`&`), OR (`|`) y NOT (`~`).

In [ ]:
%%writefile difuso/reglas.py
"""
Reglas difusas del sistema de compatibilidad.
Operadores de skfuzzy: & = AND (min), | = OR (max), ~ = NOT (1 - mu).
"""
from skfuzzy import control as ctrl


def crear_reglas(edad, afin, dist, comp):
    """Devuelve la lista de las 11 reglas difusas."""
    return [
        # FR1 afinidad alta Y distancia cerca -> alta
        ctrl.Rule(afin["alta"] & dist["cerca"], comp["alta"]),
        # FR2 muy(afinidad alta) Y NO(edad grande) -> alta
        ctrl.Rule(afin["muy_alta"] & ~edad["grande"], comp["alta"]),
        # FR3 afinidad media Y edad pequena -> media
        ctrl.Rule(afin["media"] & edad["pequena"], comp["media"]),
        # FR4 afinidad baja -> baja
        ctrl.Rule(afin["baja"], comp["baja"]),
        # FR5 distancia lejos Y NO(afinidad alta) -> baja
        ctrl.Rule(dist["lejos"] & ~afin["alta"], comp["baja"]),
        # FR6 edad grande O distancia lejos -> baja
        ctrl.Rule(edad["grande"] | dist["lejos"], comp["baja"]),
        # FR7 afinidad alta Y edad moderada -> media
        ctrl.Rule(afin["alta"] & edad["moderada"], comp["media"]),
        # FR8 afinidad media Y ligeramente(distancia lejos) -> media
        ctrl.Rule(afin["media"] & dist["ligeramente_lejos"], comp["media"]),
        # FR9 afinidad alta Y distancia lejos -> media
        ctrl.Rule(afin["alta"] & dist["lejos"], comp["media"]),
        # FR10 NO(afinidad baja) Y distancia intermedia -> media
        ctrl.Rule(~afin["baja"] & dist["intermedia"], comp["media"]),
        # FR11 afinidad media Y distancia cerca -> media
        ctrl.Rule(afin["media"] & dist["cerca"], comp["media"]),
    ]

## Interfaz y defuzzificación (centroide)

In [ ]:
%%writefile difuso/api.py
"""
Interfaz del modulo difuso que usa la integracion.
"""
import skfuzzy as fuzz
from skfuzzy import control as ctrl

from difuso.variables import crear_variables
from difuso.reglas import crear_reglas

_edad, _afin, _dist, _comp = crear_variables()
_sistema = ctrl.ControlSystem(crear_reglas(_edad, _afin, _dist, _comp))


def nivel_de(score):
    """Valor linguistico de la salida con mayor pertenencia en 'score'."""
    grados = {n: fuzz.interp_membership(_comp.universe, _comp[n].mf, score)
              for n in ("baja", "media", "alta")}
    return max(grados, key=grados.get)


def calcular_compatibilidad(edad_diff, afinidad, distancia_km):
    """Devuelve {'score': float (0-100), 'nivel': 'baja'|'media'|'alta'}.

    Las entradas se recortan al universo de cada variable antes de simular.
    """
    sim = ctrl.ControlSystemSimulation(_sistema)
    sim.input["diferencia_edad"] = min(max(edad_diff, 0), 30)
    sim.input["afinidad_intereses"] = min(max(afinidad, 0), 100)
    sim.input["distancia_km"] = min(max(distancia_km, 0), 150)
    try:
        sim.compute()
        score = float(sim.output["compatibilidad"])
    except (ValueError, KeyError):
        # ninguna regla se activo: se toma como compatibilidad nula
        score = 0.0
    return {"score": round(score, 2), "nivel": nivel_de(score)}

## Gráficas

In [ ]:
%%writefile difuso/graficas.py
"""
Graficas de las funciones de pertenencia (incluye los modificadores).
"""
import os

import matplotlib
import matplotlib.pyplot as plt

from difuso.variables import crear_variables


def graficar_funciones(carpeta="docs", mostrar=False):
    """Dibuja cada variable en un subplot y guarda la imagen en 'carpeta'."""
    variables = crear_variables()
    fig, ejes = plt.subplots(2, 2, figsize=(12, 8))
    for var, ax in zip(variables, ejes.flat):
        for nombre, termino in var.terms.items():
            estilo = "--" if nombre.startswith(("muy", "ligeramente")) else "-"
            ax.plot(var.universe, termino.mf, estilo, label=nombre)
        ax.set_title(var.label)
        ax.set_ylabel("pertenencia")
        ax.set_ylim(-0.05, 1.05)
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
    fig.tight_layout()
    os.makedirs(carpeta, exist_ok=True)
    ruta = os.path.join(carpeta, "funciones_pertenencia.png")
    fig.savefig(ruta, dpi=120)
    if mostrar:
        plt.show()
    plt.close(fig)
    return ruta


if __name__ == "__main__":
    matplotlib.use("Agg")
    print(graficar_funciones())

# Módulo 3: Sistema experto
## Clases de hechos

In [ ]:
%%writefile experto/hechos.py
"""
Clases de hechos del sistema experto.
"""
import collections
import collections.abc

# experta usa collections.Mapping, que ya no existe en Python >= 3.10
for _n in ("Mapping", "MutableMapping", "Iterable", "Callable", "Sequence"):
    if not hasattr(collections, _n):
        setattr(collections, _n, getattr(collections.abc, _n))

from experta import Fact, Field  # noqa: E402


# --- Producidos por el traductor ---

class Individuo(Fact):
    """uri es de tipo 'tipo' (triple rdf:type). origen: asertado | inferido."""
    uri = Field(str, mandatory=True)
    tipo = Field(str, mandatory=True)
    origen = Field(str, default="asertado")


class Relacion(Fact):
    """Triple cuyo objeto es otro recurso (URI)."""
    sujeto = Field(str, mandatory=True)
    predicado = Field(str, mandatory=True)
    objeto = Field(str, mandatory=True)
    origen = Field(str, default="asertado")


class Propiedad(Fact):
    """Triple cuyo objeto es un literal. valor ya viene convertido a tipo Python."""
    sujeto = Field(str, mandatory=True)
    predicado = Field(str, mandatory=True)
    valor = Field(object, mandatory=True)
    datatype = Field(str, default="")


class Esquema(Fact):
    """Axioma del esquema: subClassOf, subPropertyOf, domain o range."""
    sujeto = Field(str, mandatory=True)
    relacion = Field(str, mandatory=True)
    objeto = Field(str, mandatory=True)


# --- Producido con la logica difusa ---

class AfinidadDifusa(Fact):
    """Resultado difuso para el par (a, b), con a < b."""
    a = Field(str, mandatory=True)
    b = Field(str, mandatory=True)
    edad_diff = Field(float)
    afinidad = Field(float)
    distancia_km = Field(float)
    score = Field(float)
    nivel = Field(str)


# --- Producidos por las reglas ---

class Coincidencia(Fact):
    """Algo que el par tiene en comun. contada evita sumarla dos veces."""
    a = Field(str)
    b = Field(str)
    tipo = Field(str)
    detalle = Field(str)
    contada = Field(bool, default=False)


class Puntaje(Fact):
    """Suma de coincidencias del par."""
    a = Field(str)
    b = Field(str)
    valor = Field(int, default=0)


class Recomendacion(Fact):
    """Accion recomendada para el par."""
    a = Field(str)
    b = Field(str)
    accion = Field(str)
    motivo = Field(str, default="")


class LugarDescartado(Fact):
    """Lugar de cita donde hubo una cita mal calificada."""
    lugar = Field(str)
    motivo = Field(str, default="")


class Evaluado(Fact):
    """Lock-fact: el par ya tiene decision final."""
    a = Field(str)
    b = Field(str)

## Reglas
Salience en 5 niveles, lock-fact `Evaluado` y marca `contada` como anti-bucle.

In [ ]:
%%writefile experto/reglas.py
"""
Reglas del sistema experto de emparejamiento.

Niveles de prioridad (salience):
    ALTA = 100     descartes y creacion del puntaje
    MEDIA = 50     deteccion de coincidencias y suma del puntaje
    BAJA = 10      decision final
    DEFECTO = -10  caso por defecto
    INFORME = -20  impresion del resultado con nombres y etiquetas

Todas las reglas de decision tienen NOT(Evaluado(a, b)): Evaluado es el lock-fact
que impide que un par reciba dos decisiones finales.
"""
import vocab as V
from experto.hechos import (Individuo, Relacion, Propiedad, Esquema, AfinidadDifusa,
                            Coincidencia, Puntaje, Recomendacion, Evaluado, LugarDescartado)
from experta import Rule, NOT, TEST, MATCH, P, AS

ALTA, MEDIA, BAJA, DEFECTO, INFORME = 100, 50, 10, -10, -20


class ReglasEmparejamiento:
    """Mixin con las reglas. El motor (motor.py) hereda de esta clase."""

    def _log(self, regla, a, b, texto=""):
        """Guarda el orden de disparo y lo imprime si verbose esta activo."""
        self.disparos.append((regla, a, b))
        if self.verbose:
            print(f"[{regla}] {V.nombre_corto(a)}-{V.nombre_corto(b)} {texto}")

    def _decidir(self, regla, a, b, accion, motivo=""):
        """Declara la recomendacion final y el lock-fact Evaluado."""
        self._log(regla, a, b, f"-> {accion} {motivo}")
        self.declare(Recomendacion(a=a, b=b, accion=accion, motivo=motivo))
        self.declare(Evaluado(a=a, b=b))

    # ------------------------------------------------------------------
    # Prioridad ALTA: descartes
    # ------------------------------------------------------------------

    @Rule(AfinidadDifusa(a=MATCH.a, b=MATCH.b),
          Relacion(sujeto=MATCH.a, predicado=V.MT_BUSCA_OBJETIVO, objeto=MATCH.o1),
          Relacion(sujeto=MATCH.b, predicado=V.MT_BUSCA_OBJETIVO, objeto=MATCH.o2),
          Relacion(sujeto=MATCH.o1, predicado=V.MT_INCOMPATIBLE, objeto=MATCH.o2),
          NOT(Evaluado(a=MATCH.a, b=MATCH.b)),
          salience=ALTA)
    def r01_objetivos_incompatibles(self, a, b, o1, o2):
        """Descarta si buscan objetivos de relacion incompatibles."""
        self._decidir("R01", a, b, "descartar", "objetivos incompatibles")

    @Rule(AfinidadDifusa(a=MATCH.a, b=MATCH.b),
          Relacion(sujeto=MATCH.a, predicado=V.MT_PARTICIPA_EN, objeto=MATCH.c),
          Relacion(sujeto=MATCH.b, predicado=V.MT_PARTICIPA_EN, objeto=MATCH.c),
          Propiedad(sujeto=MATCH.c, predicado=V.MT_CALIFICACION, valor=P(lambda v: v <= 2)),
          NOT(Evaluado(a=MATCH.a, b=MATCH.b)),
          salience=ALTA)
    def r02_mala_cita_previa(self, a, b, c):
        """Descarta si ya tuvieron una cita juntos con calificacion <= 2."""
        self._decidir("R02", a, b, "descartar", "mala cita previa")

    @Rule(AfinidadDifusa(a=MATCH.a, b=MATCH.b),
          Individuo(uri=MATCH.a, tipo=V.MT_PERSONA),
          Individuo(uri=MATCH.b, tipo=V.MT_PERSONA),
          Individuo(uri=MATCH.a, tipo=V.FOAF_PERSON),
          Individuo(uri=MATCH.b, tipo=V.FOAF_PERSON),
          NOT(Puntaje(a=MATCH.a, b=MATCH.b)),
          salience=ALTA)
    def r03_iniciar_puntaje(self, a, b):
        """Crea el puntaje en 0 para cada par de personas.
        Ambos tipos (mt:Persona y foaf:Person) suelen venir del razonador."""
        self._log("R03", a, b)
        self.declare(Puntaje(a=a, b=b, valor=0))

    # ------------------------------------------------------------------
    # Prioridad MEDIA: coincidencias
    # ------------------------------------------------------------------

    @Rule(AfinidadDifusa(a=MATCH.a, b=MATCH.b),
          Relacion(sujeto=MATCH.a, predicado=V.MT_TIENE_INTERES, objeto=MATCH.x),
          Relacion(sujeto=MATCH.b, predicado=V.MT_TIENE_INTERES, objeto=MATCH.x),
          Individuo(uri=MATCH.x, tipo=V.MT_INTERES),
          NOT(Coincidencia(a=MATCH.a, b=MATCH.b, tipo="deporte", detalle=MATCH.x)),
          salience=MEDIA)
    def r04_interes_comun(self, a, b, x):
        """Interes compartido (incluye los inferidos desde practicaDeporte)."""
        self._log("R04", a, b, V.nombre_corto(x))
        self.declare(Coincidencia(a=a, b=b, tipo="interes", detalle=x))

    @Rule(AfinidadDifusa(a=MATCH.a, b=MATCH.b),
          Relacion(sujeto=MATCH.a, predicado=V.MT_TIENE_VALOR, objeto=MATCH.v),
          Relacion(sujeto=MATCH.b, predicado=V.MT_TIENE_VALOR, objeto=MATCH.v),
          Individuo(uri=MATCH.v, tipo=V.MT_VALOR),
          salience=MEDIA)
    def r05_valor_comun(self, a, b, v):
        """Valor personal compartido."""
        self._log("R05", a, b, V.nombre_corto(v))
        self.declare(Coincidencia(a=a, b=b, tipo="valor", detalle=v))

    @Rule(AfinidadDifusa(a=MATCH.a, b=MATCH.b),
          Relacion(sujeto=MATCH.a, predicado=V.MT_BUSCA_OBJETIVO, objeto=MATCH.o),
          Relacion(sujeto=MATCH.b, predicado=V.MT_BUSCA_OBJETIVO, objeto=MATCH.o),
          Individuo(uri=MATCH.o, tipo=V.MT_OBJETIVO),
          salience=MEDIA)
    def r06_mismo_objetivo(self, a, b, o):
        """Buscan el mismo tipo de relacion."""
        self._log("R06", a, b, V.nombre_corto(o))
        self.declare(Coincidencia(a=a, b=b, tipo="objetivo", detalle=o))

    @Rule(AfinidadDifusa(a=MATCH.a, b=MATCH.b),
          Relacion(sujeto=MATCH.a, predicado=V.MT_VIVE_EN, objeto=MATCH.c),
          Relacion(sujeto=MATCH.b, predicado=V.MT_VIVE_EN, objeto=MATCH.c),
          Individuo(uri=MATCH.c, tipo=V.MT_CIUDAD),
          salience=MEDIA)
    def r07_misma_ciudad(self, a, b, c):
        """Viven en la misma ciudad."""
        self._log("R07", a, b, V.nombre_corto(c))
        self.declare(Coincidencia(a=a, b=b, tipo="ciudad", detalle=c))

    @Rule(AfinidadDifusa(a=MATCH.a, b=MATCH.b),
          Relacion(sujeto=MATCH.a, predicado=V.MT_VIVE_EN, objeto=MATCH.c1),
          Relacion(sujeto=MATCH.b, predicado=V.MT_VIVE_EN, objeto=MATCH.c2),
          TEST(lambda c1, c2: c1 != c2),
          Relacion(sujeto=MATCH.c1, predicado=V.MT_UBICADO_EN, objeto=MATCH.r),
          Relacion(sujeto=MATCH.c2, predicado=V.MT_UBICADO_EN, objeto=MATCH.r),
          Individuo(uri=MATCH.r, tipo=V.MT_LUGAR),
          salience=MEDIA)
    def r07b_misma_region(self, a, b, c1, c2, r):
        """Ciudades distintas pero de la misma region."""
        self._log("R07b", a, b, V.nombre_corto(r))
        self.declare(Coincidencia(a=a, b=b, tipo="region", detalle=r))

    @Rule(AfinidadDifusa(a=MATCH.a, b=MATCH.b),
          Relacion(sujeto=MATCH.a, predicado=V.MT_VIVE_EN, objeto=MATCH.c1),
          Relacion(sujeto=MATCH.b, predicado=V.MT_VIVE_EN, objeto=MATCH.c2),
          Relacion(sujeto=MATCH.c1, predicado=V.MT_UBICADO_EN, objeto=MATCH.r1),
          Relacion(sujeto=MATCH.c2, predicado=V.MT_UBICADO_EN, objeto=MATCH.r2),
          TEST(lambda r1, r2: r1 != r2),
          Relacion(sujeto=MATCH.r1, predicado=V.MT_UBICADO_EN, objeto=MATCH.d),
          Relacion(sujeto=MATCH.r2, predicado=V.MT_UBICADO_EN, objeto=MATCH.d),
          salience=MEDIA)
    def r07c_mismo_departamento(self, a, b, c1, c2, r1, r2, d):
        """Regiones distintas dentro del mismo lugar mayor (departamento)."""
        self._log("R07c", a, b, V.nombre_corto(d))
        self.declare(Coincidencia(a=a, b=b, tipo="departamento", detalle=d))

    @Rule(AfinidadDifusa(a=MATCH.a, b=MATCH.b),
          Relacion(sujeto=MATCH.a, predicado=V.FOAF_KNOWS, objeto=MATCH.c),
          Relacion(sujeto=MATCH.b, predicado=V.FOAF_KNOWS, objeto=MATCH.c),
          TEST(lambda a, b, c: c not in (a, b)),
          salience=MEDIA)
    def r08_amigo_comun(self, a, b, c):
        """Conocen a la misma persona (foaf:knows se infiere desde esAmigoDe)."""
        self._log("R08", a, b, V.nombre_corto(c))
        self.declare(Coincidencia(a=a, b=b, tipo="amigo_comun", detalle=c))

    @Rule(AfinidadDifusa(a=MATCH.a, b=MATCH.b),
          Individuo(uri=MATCH.a, tipo=MATCH.t),
          Individuo(uri=MATCH.b, tipo=MATCH.t),
          Esquema(sujeto=MATCH.t, relacion=V.SUBCLASE, objeto=V.MT_PERSONA),
          Esquema(sujeto=MATCH.t, relacion=V.TIPO, objeto=V.RDFS_CLASS),
          TEST(lambda t: t != V.MT_PERSONA),
          salience=MEDIA)
    def r09_misma_etapa(self, a, b, t):
        """Misma subclase de Persona (se lee del esquema, no esta fija en el codigo).
        Se exige ademas que t este declarada como rdfs:Class."""
        self._log("R09", a, b, V.nombre_corto(t))
        self.declare(Coincidencia(a=a, b=b, tipo="etapa", detalle=t))

    @Rule(AfinidadDifusa(a=MATCH.a, b=MATCH.b),
          Relacion(sujeto=MATCH.a, predicado=V.MT_PRACTICA_DEPORTE, objeto=MATCH.d),
          Relacion(sujeto=MATCH.b, predicado=V.MT_PRACTICA_DEPORTE, objeto=MATCH.d),
          Individuo(uri=MATCH.d, tipo=V.MT_DEPORTE),
          Individuo(uri=MATCH.d, tipo=V.MT_INTERES),
          salience=MEDIA)
    def r10_deporte_comun(self, a, b, d):
        """Deporte en comun. Es mas especifica que R04 y le gana por specificity.
        Deporte es Interes por inferencia (subClassOf), por eso tambien se pide ese tipo."""
        self._log("R10", a, b, V.nombre_corto(d))
        self.declare(Coincidencia(a=a, b=b, tipo="deporte", detalle=d))

    @Rule(AS.c << Coincidencia(a=MATCH.a, b=MATCH.b, tipo=MATCH.t, contada=False),
          AS.p << Puntaje(a=MATCH.a, b=MATCH.b, valor=MATCH.v),
          salience=MEDIA)
    def r11_sumar_puntaje(self, c, p, a, b, t, v):
        """Suma la coincidencia al puntaje (+1, deporte +2).

        Esta regla modifica un hecho que esta en su propia condicion (Puntaje),
        asi que se marca contada=True para no volver a sumarla (anti-bucle).
        """
        puntos = 2 if t == "deporte" else 1
        self.modify(c, contada=True)
        self.modify(p, valor=v + puntos)
        self._log("R11", a, b, f"+{puntos} ({t}) = {v + puntos}")

    # ------------------------------------------------------------------
    # Prioridad BAJA: decision final
    # ------------------------------------------------------------------

    @Rule(AfinidadDifusa(a=MATCH.a, b=MATCH.b, nivel="alta"),
          Puntaje(a=MATCH.a, b=MATCH.b, valor=P(lambda v: v >= 3)),
          Propiedad(sujeto=MATCH.a, predicado=V.MT_VERIFICADO, valor=True),
          Propiedad(sujeto=MATCH.b, predicado=V.MT_VERIFICADO, valor=True),
          NOT(Evaluado(a=MATCH.a, b=MATCH.b)),
          salience=BAJA)
    def r12_proponer_cita(self, a, b):
        """Compatibilidad alta, buen puntaje y ambos verificados (la mas especifica)."""
        self._decidir("R12", a, b, "proponer_cita")

    @Rule(AfinidadDifusa(a=MATCH.a, b=MATCH.b, nivel=P(lambda n: n in ("media", "alta"))),
          Puntaje(a=MATCH.a, b=MATCH.b, valor=P(lambda v: v >= 2)),
          NOT(Evaluado(a=MATCH.a, b=MATCH.b)),
          salience=BAJA)
    def r13_sugerir_conversar(self, a, b):
        """Regla general: compatibilidad media o alta y puntaje de al menos 2."""
        self._decidir("R13", a, b, "sugerir_conversar")

    @Rule(AfinidadDifusa(a=MATCH.a, b=MATCH.b, nivel="alta"),
          Puntaje(a=MATCH.a, b=MATCH.b),
          Propiedad(sujeto=MATCH.x, predicado=V.MT_VERIFICADO, valor=False),
          TEST(lambda x, a, b: x in (a, b)),
          NOT(Evaluado(a=MATCH.a, b=MATCH.b)),
          salience=BAJA)
    def r14_pedir_verificacion(self, a, b, x):
        """Compatibilidad alta pero alguno no tiene el perfil verificado."""
        self._decidir("R14", a, b, "pedir_verificacion", V.nombre_corto(x))

    @Rule(AfinidadDifusa(a=MATCH.a, b=MATCH.b, nivel="baja"),
          NOT(Evaluado(a=MATCH.a, b=MATCH.b)),
          salience=BAJA)
    def r15_compatibilidad_baja(self, a, b):
        """La logica difusa dio compatibilidad baja."""
        self._decidir("R15", a, b, "descartar", "compatibilidad baja")

    @Rule(Relacion(sujeto=MATCH.c, predicado=V.MT_OCURRIO_EN, objeto=MATCH.l),
          Individuo(uri=MATCH.c, tipo=V.MT_CITA),
          Propiedad(sujeto=MATCH.c, predicado=V.MT_CALIFICACION, valor=P(lambda v: v <= 2)),
          salience=MEDIA)
    def r16a_marcar_lugar(self, c, l):
        """Marca los lugares donde hubo una cita con calificacion <= 2."""
        self._log("R16a", l, c)
        self.declare(LugarDescartado(lugar=l, motivo=c))

    @Rule(Recomendacion(a=MATCH.a, b=MATCH.b, accion="proponer_cita"),
          Relacion(sujeto=MATCH.a, predicado=V.MT_VIVE_EN, objeto=MATCH.c),
          Relacion(sujeto=MATCH.b, predicado=V.MT_VIVE_EN, objeto=MATCH.c),
          Relacion(sujeto=MATCH.l, predicado=V.MT_UBICADO_EN, objeto=MATCH.c),
          Individuo(uri=MATCH.l, tipo=V.MT_LUGAR_CITA),
          NOT(LugarDescartado(lugar=MATCH.l)),
          NOT(Recomendacion(a=MATCH.a, b=MATCH.b, accion="sugerir_lugar")),
          salience=BAJA)
    def r16_sugerir_lugar(self, a, b, c, l):
        """Sugiere un lugar de la ciudad de ambos donde no haya habido citas malas."""
        self._log("R16", a, b, V.nombre_corto(l))
        self.declare(Recomendacion(a=a, b=b, accion="sugerir_lugar", motivo=l))

    @Rule(Puntaje(a=MATCH.a, b=MATCH.b),
          NOT(Evaluado(a=MATCH.a, b=MATCH.b)),
          salience=DEFECTO)
    def r17_revisar_manual(self, a, b):
        """Ninguna regla de decision aplico: queda para revision manual."""
        self._decidir("R17", a, b, "revisar_manual")

    # ------------------------------------------------------------------
    # Control de consistencia con el esquema y reporte
    # ------------------------------------------------------------------

    @Rule(Relacion(sujeto=MATCH.s, predicado=MATCH.p, objeto=MATCH.o),
          Esquema(sujeto=MATCH.p, relacion=V.RANGO, objeto=MATCH.c),
          NOT(Individuo(uri=MATCH.o, tipo=MATCH.c)),
          salience=ALTA)
    def r18_validar_rango(self, s, p, o, c):
        """Avisa si un objeto no es del tipo que pide el rango de la propiedad.
        Con el grafo razonado no deberia dispararse nunca."""
        print(f"[R18] Inconsistencia: {V.nombre_corto(o)} no es {V.nombre_corto(c)}")

    @Rule(Recomendacion(a=MATCH.a, b=MATCH.b, accion=MATCH.acc, motivo=MATCH.m),
          TEST(lambda acc: acc != "sugerir_lugar"),
          Propiedad(sujeto=MATCH.a, predicado=V.FOAF_NAME, valor=MATCH.na),
          Propiedad(sujeto=MATCH.b, predicado=V.FOAF_NAME, valor=MATCH.nb),
          salience=INFORME)
    def r19_informe(self, a, b, acc, m, na, nb):
        """Guarda el resultado del par con los nombres (foaf:name)."""
        self.informe.setdefault((a, b), {"nombres": (na, nb)})
        self.informe[(a, b)]["accion"] = acc
        self.informe[(a, b)]["motivo"] = m

    @Rule(Recomendacion(a=MATCH.a, b=MATCH.b, accion="sugerir_lugar", motivo=MATCH.l),
          Propiedad(sujeto=MATCH.l, predicado=V.RDFS_LABEL, valor=MATCH.etiqueta),
          salience=INFORME)
    def r20_informe_lugar(self, a, b, l, etiqueta):
        """Agrega al informe el lugar sugerido usando su rdfs:label."""
        self.informe.setdefault((a, b), {})["lugar"] = str(etiqueta)

## Motor y estrategia de resolución de conflictos (salience, specificity, recencia)

In [ ]:
%%writefile experto/motor.py
"""
Motor de inferencia y estrategia de resolucion de conflictos.
"""
from experto.hechos import Recomendacion  # importa primero: aplica el parche de collections
from experta import KnowledgeEngine
from experta.strategies import DepthStrategy

from experto.reglas import ReglasEmparejamiento


class EstrategiaEmparejamiento(DepthStrategy):
    """Ordena la agenda por (salience, specificity, recencia).

    - salience: prioridad declarada en la regla.
    - specificity: cantidad de hechos que empareja la activacion; a igual
      salience gana la regla con mas condiciones (la mas especifica).
    - recencia: ids de los hechos, de mayor a menor; a igualdad de lo anterior
      gana la activacion con hechos declarados mas recientemente.
    Experta solo trae salience + recencia (DepthStrategy), la specificity se agrega aqui.
    """

    def get_key(self, activation):
        """Llave con la que se ordena la agenda (la mayor se ejecuta primero)."""
        salience = activation.rule.salience
        especificidad = len(activation.facts)
        recencia = sorted((f["__factid__"] for f in activation.facts), reverse=True)
        return (salience, especificidad, recencia)


class MotorEmparejamiento(ReglasEmparejamiento, KnowledgeEngine):
    """Motor de Experta con las reglas de emparejamiento."""

    __strategy__ = EstrategiaEmparejamiento

    def __init__(self, verbose=False):
        """verbose=True imprime cada regla al dispararse."""
        super().__init__()
        self.verbose = verbose
        self.disparos = []
        self.informe = {}

    def cargar(self, facts):
        """Reinicia el motor y declara los hechos iniciales en el orden recibido."""
        self.reset()
        self.disparos = []
        self.informe = {}
        for f in facts:
            self.declare(f)

    def recomendaciones(self):
        """Lista de Facts Recomendacion que hay en la memoria de trabajo."""
        return [f for f in self.facts.values() if isinstance(f, Recomendacion)]

# Integración
## Traductor Ontología → Sistema experto
No contiene nombres del dominio: solo usa los IRIs de RDF, RDFS, OWL y XSD.

In [ ]:
%%writefile integracion/traductor.py
"""
Traductor Ontologia -> Sistema experto.

Recorre el grafo ya razonado y convierte cada triple en un Fact generico.
No conoce ningun nombre del dominio: solo usa los IRIs de RDF, RDFS, OWL y XSD,
asi que si se agrega algo al .ttl aparece solo en el motor.
"""
from decimal import Decimal

from rdflib import RDF, RDFS, OWL, XSD, Literal, URIRef

from experto.hechos import Individuo, Relacion, Propiedad, Esquema

# namespaces que se consideran "infraestructura" y no conocimiento del dominio
NS_INFRA = (str(RDF), str(RDFS), str(OWL), str(XSD))

# predicados de RDFS que si describen el esquema del dominio
PRED_ESQUEMA = {
    RDFS.subClassOf: "subClassOf",
    RDFS.subPropertyOf: "subPropertyOf",
    RDFS.domain: "domain",
    RDFS.range: "range",
}

# declaraciones de clase y propiedad (s rdf:type rdfs:Class / rdf:Property)
DECLARACIONES = (RDFS.Class, RDF.Property)


def es_infra(nodo):
    """True si el nodo pertenece a RDF/RDFS/OWL/XSD."""
    return isinstance(nodo, URIRef) and str(nodo).startswith(NS_INFRA)


def traducir(g_inferido, g_original):
    """Traduce el grafo inferido a Facts genericos.

    'origen' = 'inferido' si el triple no esta en g_original, si no 'asertado'.
    """
    facts = []
    # se recorre ordenado para que el orden de los hechos (y la recencia) sea
    # el mismo en todas las ejecuciones
    for s, p, o in sorted(g_inferido):
        # sujetos literales/blank nodes o de infraestructura no aportan al dominio
        if not isinstance(s, URIRef) or es_infra(s):
            continue
        origen = "asertado" if (s, p, o) in g_original else "inferido"

        if p == RDF.type:
            if o in DECLARACIONES:
                facts.append(Esquema(sujeto=str(s), relacion="tipo", objeto=str(o)))
            elif not es_infra(o):
                facts.append(Individuo(uri=str(s), tipo=str(o), origen=origen))
        elif p in PRED_ESQUEMA:
            if isinstance(o, URIRef) and not str(o).startswith(NS_INFRA[:3]):
                facts.append(Esquema(sujeto=str(s), relacion=PRED_ESQUEMA[p], objeto=str(o)))
        elif isinstance(o, Literal):
            if es_infra(p) and p != RDFS.label:
                continue
            tipo = str(o.datatype) if o.datatype else ""
            valor = o.toPython()
            if isinstance(valor, Decimal):
                valor = float(valor)
            facts.append(Propiedad(sujeto=str(s), predicado=str(p),
                                   valor=valor, datatype=tipo))
        elif isinstance(o, URIRef) and not es_infra(p):
            facts.append(Relacion(sujeto=str(s), predicado=str(p),
                                  objeto=str(o), origen=origen))
    return facts


def resumen(facts):
    """Cuenta cuantos Facts hay de cada clase y cuantos son inferidos."""
    conteo = {}
    for f in facts:
        nombre = type(f).__name__
        conteo.setdefault(nombre, [0, 0])
        conteo[nombre][0] += 1
        if f.get("origen") == "inferido":
            conteo[nombre][1] += 1
    return conteo

## Preparación de las entradas del difuso

In [ ]:
%%writefile integracion/preparar_difuso.py
"""
Prepara las entradas del difuso para cada par de personas y crea los AfinidadDifusa.
"""
from itertools import combinations
from math import radians, sin, cos, asin, sqrt

import vocab as V
from difuso.api import calcular_compatibilidad
from experto.hechos import Individuo, Relacion, Propiedad, AfinidadDifusa


def haversine(lat1, lon1, lat2, lon2):
    """Distancia en km entre dos coordenadas geograficas."""
    lat1, lon1, lat2, lon2 = map(radians, (lat1, lon1, lat2, lon2))
    h = sin((lat2 - lat1) / 2) ** 2 + cos(lat1) * cos(lat2) * sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371 * asin(sqrt(h))


def jaccard(i_a, i_b):
    """Afinidad en % como |A n B| / |A u B|. Si ambos estan vacios es 0."""
    union = i_a | i_b
    return 100 * len(i_a & i_b) / len(union) if union else 0.0


def calcular_pares(facts):
    """Para cada par no ordenado de Individuo(tipo=Persona), calcula entradas y llama a difuso."""
    personas = sorted({f["uri"] for f in facts
                       if isinstance(f, Individuo) and f["tipo"] == V.MT_PERSONA})

    intereses, ciudad, edad, coords = {}, {}, {}, {}
    for f in facts:
        if isinstance(f, Relacion):
            if f["predicado"] == V.MT_TIENE_INTERES:
                intereses.setdefault(f["sujeto"], set()).add(f["objeto"])
            elif f["predicado"] == V.MT_VIVE_EN:
                ciudad[f["sujeto"]] = f["objeto"]
        elif isinstance(f, Propiedad):
            if f["predicado"] == V.MT_EDAD:
                edad[f["sujeto"]] = f["valor"]
            elif f["predicado"] == V.MT_LATITUD:
                coords.setdefault(f["sujeto"], [0, 0])[0] = f["valor"]
            elif f["predicado"] == V.MT_LONGITUD:
                coords.setdefault(f["sujeto"], [0, 0])[1] = f["valor"]

    resultado = []
    for a, b in combinations(personas, 2):
        # si falta la edad o la ciudad se toma el peor caso del universo
        edad_diff = abs(edad[a] - edad[b]) if a in edad and b in edad else 30.0
        afin = jaccard(intereses.get(a, set()), intereses.get(b, set()))
        ca, cb = ciudad.get(a), ciudad.get(b)
        if ca in coords and cb in coords:
            dist = haversine(*coords[ca], *coords[cb])
        else:
            dist = 150.0
        r = calcular_compatibilidad(edad_diff, afin, dist)
        resultado.append(AfinidadDifusa(a=a, b=b, edad_diff=float(edad_diff),
                                        afinidad=float(round(afin, 2)), distancia_km=float(round(dist, 2)),
                                        score=float(r["score"]), nivel=r["nivel"]))
    return resultado

## Programa principal

In [ ]:
%%writefile main.py
"""
Sistema hibrido de emparejamiento: Ontologia + Logica difusa + Sistema experto.

Flujo:
    1. Se carga el .ttl y se razona con DeductiveClosure(RDFS_Semantics).
    2. El traductor convierte todo el grafo inferido en Facts.
    3. Para cada par de personas se calcula la compatibilidad difusa.
    4. El motor experto recibe los Facts + AfinidadDifusa y decide.

Uso:  python main.py [ruta_ttl]
"""
import os
import sys

import vocab as V
from ontologia.razonamiento import cargar_y_razonar, mostrar_razonamiento
from integracion.traductor import traducir, resumen
from integracion.preparar_difuso import calcular_pares
from experto.hechos import Individuo
from experto.motor import MotorEmparejamiento

BASE = os.path.dirname(os.path.abspath(__file__))
RUTA_TTL = os.path.join(BASE, "ontologia", "emparejamiento.ttl")


def titulo(texto):
    """Imprime un encabezado de seccion."""
    print("\n" + "=" * 70 + f"\n{texto}\n" + "=" * 70)


def ejecutar(ruta_ttl=RUTA_TTL, pares_traza=(("Andres", "Sofia"), ("Mateo", "Sofia"))):
    """Corre todo el sistema y devuelve el motor ya ejecutado."""
    titulo("1. ONTOLOGIA Y RAZONAMIENTO RDFS")
    g_original, g_inferido = cargar_y_razonar(ruta_ttl)
    mostrar_razonamiento(g_original, g_inferido)

    titulo("2. TRADUCTOR ONTOLOGIA -> HECHOS")
    facts = traducir(g_inferido, g_original)
    print(f"Hechos generados: {len(facts)}")
    for clase, (total, inferidos) in resumen(facts).items():
        print(f"   {clase:<10} {total:>4}  (inferidos: {inferidos})")
    print("Personas encontradas y sus tipos:")
    tipos = {}
    for f in facts:
        if isinstance(f, Individuo):
            tipos.setdefault(f["uri"], []).append(f"{V.nombre_corto(f['tipo'])} ({f['origen']})")
    for uri in sorted(tipos):
        if any(t.startswith("Persona ") for t in tipos[uri]):
            print(f"   {V.nombre_corto(uri):<10} " + ", ".join(sorted(tipos[uri])))

    titulo("3. LOGICA DIFUSA POR PAR")
    afinidades = calcular_pares(facts)
    print(f"{'par':<20}{'edad':>6}{'afin%':>8}{'km':>8}{'score':>8}  nivel")
    for af in afinidades:
        par = f"{V.nombre_corto(af['a'])}-{V.nombre_corto(af['b'])}"
        print(f"{par:<20}{af['edad_diff']:>6.0f}{af['afinidad']:>8.1f}"
              f"{af['distancia_km']:>8.1f}{af['score']:>8.1f}  {af['nivel']}")

    titulo("4. SISTEMA EXPERTO")
    motor = MotorEmparejamiento()
    motor.cargar(facts + afinidades)
    motor.run()

    print("Primeros disparos del motor (a igual salience y specificity gana el hecho mas reciente):")
    for regla, x, y in motor.disparos[:8]:
        print(f"   {regla} {V.nombre_corto(x)}-{V.nombre_corto(y)}")

    for a, b in pares_traza:
        ua, ub = V.MT + a, V.MT + b
        print(f"\nOrden de disparo para {a}-{b}:")
        for regla, x, y in motor.disparos:
            if (x, y) == (ua, ub):
                print(f"   {regla}", end="")
        print()

    # se recorren los Facts Recomendacion; el informe (R19/R20) solo aporta nombres
    print("\nRecomendaciones:")
    lugares = {(r["a"], r["b"]): r["motivo"] for r in motor.recomendaciones()
               if r["accion"] == "sugerir_lugar"}
    conteo = {}
    for r in sorted(motor.recomendaciones(), key=lambda f: (f["a"], f["b"])):
        if r["accion"] == "sugerir_lugar":
            continue
        a, b = r["a"], r["b"]
        info = motor.informe.get((a, b), {})
        na, nb = info.get("nombres", (V.nombre_corto(a), V.nombre_corto(b)))
        linea = f"   {na + ' - ' + nb:<22} {r['accion']:<20} {r['motivo']}"
        if (a, b) in lugares:
            linea += f"  lugar sugerido: {info.get('lugar', V.nombre_corto(lugares[(a, b)]))}"
        print(linea)
        conteo[r["accion"]] = conteo.get(r["accion"], 0) + 1

    print(f"\nTotal de pares: {sum(conteo.values())}  {conteo}")
    return motor


if __name__ == "__main__":
    ejecutar(sys.argv[1] if len(sys.argv) > 1 else RUTA_TTL)

# Ejecución

In [ ]:
!python main.py

## Gráficas para el documento

In [ ]:
!python -m difuso.graficas
!python -m ontologia.diagrama
from IPython.display import Image, display
display(Image("docs/funciones_pertenencia.png"))
display(Image("docs/diagrama_clases.png"))

# Prueba obligatoria: agregar un individuo solo en el `.ttl`
Se agrega a Valentina directamente al archivo de la ontología, sin tocar ningún archivo de Python, y se vuelve a ejecutar.
Debe aparecer como `Individuo` (Estudiante asertado y Persona inferido), con sus pares difusos y sus recomendaciones.

In [ ]:
%%writefile -a ontologia/emparejamiento.ttl

mt:Valentina a mt:Estudiante ;
    foaf:name "Valentina" ; mt:edad 24 ;
    mt:viveEn mt:Medellin ;
    mt:tieneInteres mt:Cine ; mt:practicaDeporte mt:Running ;
    mt:tieneValor mt:Honestidad ; mt:buscaObjetivo mt:RelacionSeria ;
    mt:perfilVerificado true .

In [ ]:
!python main.py | grep -E "Valentina|Total|Hechos generados"